# selfRag8

In [ ]:

from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.tools.tavily_search import TavilySearchResults


from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()
True
docs = (
    PyPDFLoader("./documents/Company_Policies.pdf").load()
    + PyPDFLoader("./documents/Company_Profile.pdf").load()
    + PyPDFLoader("./documents/Product_and_Pricing.pdf").load()
)
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=150
).split_documents(docs)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
class State(TypedDict):
    question: str
    need_retrieval: bool

    docs: List[Document]
    relevant_docs: List[Document]

    context: str
    answer: str

    # web query (no loop flags)
    web_query: str
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
            "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}"),
    ]
)


# IMPORTANT: no `.content` for structured output
should_retrieve_llm = llm.with_structured_output(RetrieveDecision)

def decide_retrieval(state: "State"):
    decision: RetrieveDecision = should_retrieve_llm.invoke(
        decide_retrieval_prompt.format_messages(question=state["question"])
    )
    return {"need_retrieval": decision.should_retrieve}
direct_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human", "{question}"),
    ]
)


def generate_direct(state: State):
    out = llm.invoke(
        direct_generation_prompt.format_messages(
            question=state["question"]
        )
    )
    return {
        "answer": out.content
    }
def retrieve(state: State):
    return {"docs": retriever.invoke(state["question"])}
class RelevanceDecision(BaseModel):
    is_relevant: bool = Field(
        ...,
        description="True if the document helps answer the question, else False."
    )

is_relevant_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are judging document relevance.\n"
            "Return JSON that matches this schema:\n"
            "{{'is_relevant': boolean}}\n\n"
            "A document is relevant if it contains information useful for answering the question."
        ),
        (
            "human",
            "Question:\n{question}\n\nDocument:\n{document}"
        ),
    ]
)

relevance_llm = llm.with_structured_output(RelevanceDecision)

def is_relevant(state: State):
    
    relevant_docs: List[Document] = []

    for doc in state["docs"]:
        decision: RelevanceDecision = relevance_llm.invoke(
            is_relevant_prompt.format_messages(
                question=state["question"],
                document=doc.page_content
            )
        )

        if decision.is_relevant:
            relevant_docs.append(doc)

    return {"relevant_docs": relevant_docs}
# New
rag_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a business RAG assistant.\n"
            "Answer the user's question using ONLY the provided context.\n"
            "If the context does not contain enough information, say:\n"
            "'No relevant document found.'\n"
            "Do not use outside knowledge.\n"
        ),
        (
            "human",
            "Question:\n{question}\n\n"
            "Context:\n{context}\n"
        ),
    ]
)

def generate_from_context(state: State):
    # Stuff relevant docs into one block
    context = "\n\n---\n\n".join(
        [d.page_content for d in state.get("relevant_docs", [])]
    ).strip()

    if not context:
        return {"answer": "No relevant document found.", "context": ""}

    out = llm.invoke(
        rag_generation_prompt.format_messages(
            question=state["question"],
            context=context
        )
    )
    return {"answer": out.content, "context": context}
class WebQuery(BaseModel):
    query: str

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user question into a web search query composed of keywords.\n"
            "Rules:\n"
            "- Keep it short (6–14 words).\n"
            "- If the question implies recency, add (last 30 days).\n"
            "- Do NOT answer the question.\n"
            "- Return JSON with a single key: query",
        ),
        ("human", "Question: {question}"),
    ]
)

rewrite_chain = rewrite_prompt | llm.with_structured_output(WebQuery)

def rewrite_query_node(state: State):
    out = rewrite_chain.invoke({"question": state["question"]})
    return {"web_query": out.query}

tavily = TavilySearchResults(max_results=5)

def web_search_node(state: State):
    q = state.get("web_query") or state["question"]
    results = tavily.invoke({"query": q})

    docs = []
    for r in results or []:
        title = r.get("title", "")
        url = r.get("url", "")
        content = r.get("content", "") or r.get("snippet", "")
        text = f"TITLE: {title}\nURL: {url}\nCONTENT:\n{content}"
        docs.append(
            Document(
                page_content=text,
                metadata={"source": "web", "url": url, "title": title},
            )
        )

    return {"docs": docs}
/var/folders/h1/gh59pbb174b94mbxsb8xb9qc0000gn/T/ipykernel_87698/948566410.py:25: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(max_results=5)
# New
def no_relevant_docs(state: State):
    return {"answer": "No relevant document found.", "context": ""}
def route_after_decide(state: State) -> Literal["generate_direct", "retrieve"]:
    if state["need_retrieval"]:
        return "retrieve"
    return "generate_direct"
def route_after_relevance(state: State) -> Literal["generate_from_context", "rewrite_query"]:
    if state.get("relevant_docs") and len(state["relevant_docs"]) > 0:
        return "generate_from_context"
    return "rewrite_query"
g = StateGraph(State)

g.add_node("decide_retrieval", decide_retrieval)
g.add_node("generate_direct", generate_direct)
g.add_node("retrieve", retrieve)

g.add_node("is_relevant", is_relevant)
g.add_node("generate_from_context", generate_from_context)

# ✅ new nodes (replace no_relevant_docs)
g.add_node("rewrite_query", rewrite_query_node)
g.add_node("web_search", web_search_node)

# --------------------
# Edges
# --------------------
g.add_edge(START, "decide_retrieval")

g.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {
        "generate_direct": "generate_direct",
        "retrieve": "retrieve",
    },
)

g.add_edge("generate_direct", END)

# vector retrieval → relevance
g.add_edge("retrieve", "is_relevant")

# relevance router: if relevant → generate, else → rewrite_query
g.add_conditional_edges(
    "is_relevant",
    route_after_relevance,
    {
        "generate_from_context": "generate_from_context",
        "rewrite_query": "rewrite_query",
    },
)

# web fallback path
g.add_edge("rewrite_query", "web_search")
g.add_edge("web_search", "is_relevant")  # 🔁 circle back

# final
g.add_edge("generate_from_context", END)

app = g.compile()
app

result = app.invoke(
    {
        "question": "Who won the Aus vs Zim World T20 match 2026 and who was the top scorer",
        "docs": [],
        "relevant_docs": [],
        "context": "",
        "answer": "",
    }
)

print(result["answer"])
Zimbabwe won the Aus vs Zim World T20 match in 2026 by 23 runs. The top scorer for Zimbabwe was Brian Bennett, who scored an unbeaten 64 runs. For Australia, the top scorer was Matt Renshaw with 65 runs.
for doc in result['relevant_docs']:
    print(doc.page_content)
    print("*"*100)
TITLE: Australia vs Zimbabwe, ICC Men's T20 World Cup 2026, 19th Match ...
URL: https://www.espn.com/cricket/series/8604/game/1512737/australia-vs-zimbabwe-19th-match-group-b-8604
CONTENT:
2.55pm Right then, we officially have the first upset of the 2026 T20 World Cup, with Zimbabwe's statement win over Australia. It's been an incredible game of cricket, and it opens Group B up for a thrilling conclusion. Sri Lanka have to play both Australia and Zimbabwe, and right now, it is hard to tell who will progress from this group. Zimbabwe will celebrate a win for the ages before recalibrating for their next game. Australia will hope for reinforcements to arrive swiftly, with more injury worries after today's game. That's all from the Premadasa from us today. Next on the menu is Canada taking on UAE in Delhi, where Canada will be batting first. Head on over there for more T20 World Cup action. From this game, it's Abhimanyu signing off on behalf of Ranjith, and everyone else at [...] signing off on behalf of Ranjith, and everyone else at ESPNcricinfo. [...] Sikandar Raza: "All I can tell you is that these are just cramps and I should be fine in the next day or two. Very happy and above all, very proud. Feeling of a brother whose younger brothers are achieving a lot together. The culture, environment, and unity we've created is amazing and on top of that, to win is unbelievable. 70-odd at 10 overs and I was happy and we discussed that we don't want to go for 190. We try to go for 190, we are 140 all out. We've seen in Sri Lanka that you can end up losing wickets if you go too hard too early. So we sent a message to Benny that he's doing well and we'll get to a good score if he stays through. And then the fielding, the catching, the boundary stopping ... looked like the boys really wanted it. [Injuries] We have just 13 able bodies right now
****************************************************************************************************
TITLE: Historic Win! Zimbabwe Defeat Australia in T20 World Cup 2026
URL: https://www.youtube.com/watch?v=ivIoGKcQe4o
CONTENT:
दोस्तों T20 वर्ल्ड कप 2026 में अब तक कई मुकाबले क्लोज गए हैं। लेकिन आज पहली बार मतलब एसोसिएट नेशन है, छोटी टीमें हैं। वो पहली बार जो है मुकाबला जीतती हुई दिखी है। मतलब क्लोज ही नहीं ले गए। बीत भी गया है। मतलब अपसेट कर दिया है ऑस्ट्रेलिया को। हरा दिया है T20 वर्ल्ड कप के मुकाबले में और जिंबाब्वे ने यह काम पहली बार नहीं किया है। मतलब हिस्ट्री रिपीट कर रहे हैं जिंबाब्वे के जो खिलाड़ी है बहुत बड़ी जीत है जिंबाब्वे के लिए। जिस तरीके से उनकी बल्लेबाजी हुई। शुरुआत हुई तो लग रहा था बहुत धीमी पारी चल रही थी वहां पे। लेकिन बहुत समझदारी भरी पारी थी और उतने ही रन काफी बड़े ऑस्ट्रेलिया के लिए। मतलब 20 ओवर में निर्धारित 20 ओवर में 169 रन बनाते हैं जिंबाब्वे के खिलाड़ी और इस दौरान उनके सिर्फ दो ही विकेट दो ही खिलाड़ी आउट होते हैं। तो लग रहा था कि थोड़े से थोड़े से रन पीछे रह गई है जिंबाब्वे लेकिन जब [...] ### Transcript: [...] # Historic Win! Zimbabwe Defeat Australia in T20 World Cup 2026 | Full Match Review
## Cricket Times
1030000 subscribers
4 likes

### Description
387 views
Posted: 13 Feb 2026
Zimbabwe Create History in T20 World Cup 2026! 🇿🇼🔥

In one of the biggest upsets of the tournament, Zimbabwe stunned Australia with a historic victory in the T20 World Cup 2026. No one expected this result, but Zimbabwe showed fearless cricket, smart captaincy, and incredible team effort to defeat the mighty Aussies.

In this video, we break down:

Key turning points of the match

Top performers from both teams

Where Australia went wrong

What this win means for Zimbabwe in the tournament

Points table impact & qualification scenario
****************************************************************************************************
TITLE: Australia vs Zimbabwe LIVE: ICC T20 World Cup 2026 - BBC
URL: https://www.bbc.com/sport/cricket/live/cly1nvmzrdgt
CONTENT:
| Total,Total169 for 2 20.0 overs  Total,Total169 for 2 20.0 overs | 169-2 | [...] | Fall of wicket | Batter |
 --- |
| 13 for 113-1 (1.1 overs) | Inglis |
| 24 for 224-2 (2.5 overs) | Green |
| 25 for 325-3 (3.2 overs) | David |
| 29 for 429-4 (4.3 overs) | Head |
| 106 for 5106-5 (14.2 overs) | Maxwell |
| 117 for 6117-6 (15.4 overs) | Stoinis |
| 131 for 7131-7 (17.4 overs) | Dwarshuis |
| 139 for 8139-8 (18.4 overs) |  |
| 141 for 9141-9 (18.6 overs) | Zampa |
| 146 for 10146-10 (19.3 overs) | Kuhnemann |

## Team Lineups

### home team, Australia

#### Starting lineup

 Travis Head (c), Captain
 Josh Inglis (wk), Wicket Keeper
 Cameron Green
 Tim David
 Matt Renshaw
 Glenn Maxwell
 Marcus Stoinis
 Nathan Ellis
 Adam Zampa
 Matt Kuhnemann
 Ben Dwarshuis

### away team, Zimbabwe

#### Starting lineup [...] | Maxwell,Maxwellbowled Burl b Burl  Maxwell,Maxwell bowled Burl b Burl | 31 | 32 | 12 | 1 | 1 | 59 | 96.88 |
| Renshaw,Renshawcaught Burl, bowled Muzarabani c Burl  b Muzarabani  Renshaw,Renshaw caught Burl, bowled Muzarabani c Burl  b Muzarabani | 65 | 44 | 7 | 5 | 1 | 74 | 147.73 |
| Stoinis,Stoiniscaught Musekiwa, bowled Masakadza c Musekiwa  b Masakadza  Stoinis,Stoinis caught Musekiwa, bowled Masakadza c Musekiwa  b Masakadza | 6 | 4 | 1 | 1 | 0 | 6 | 150.00 |
| Dwarshuis,Dwarshuiscaught Munyonga, bowled Evans c Munyonga  b Evans  Dwarshuis,Dwarshuis caught Munyonga, bowled Evans c Munyonga  b Evans | 6 | 7 | 2 | 0 | 0 | 9 | 85.71 |
| Ellis,Ellisnot out not out  Ellis,Ellis not out not out | 7 | 4 | 0 | 1 | 0 | 11 | 175.00 |
****************************************************************************************************
TITLE: USA, UAE triumphant, Zimbabwe script historic T20WC win
URL: https://www.icc-cricket.com/tournaments/mens-t20-world-cup-2026/news/live-australia-look-to-stay-unbeaten-at-t20-world-cup
CONTENT:
Unbeaten in the tournament, they are second in Group B standings while Australia have slipped to three. Only top two players from each group will progress to the Super 8 stage.

Australia won the toss and elected to field, but little went their way after that. In an assured batting performance, posted 169/2 in 20 overs, their highest score against Australia in T20Is. Zimbabwe's previous best against Australia was 151/9 in Harare in July 2018.

They carried on the momentum in the field, reducing Australia to 29/4 inside the first five overs.

Zimbabwe script huge win in Colombo | T20WC 2026

Zimbabwe rise to the occasion with a brilliant all-round display to upset Australia by 23 runs at the ICC Men's T20 World Cup 2026. [...] With the Zimbabwe batters taking on the challenge, Australia tried out seven bowlers in the first 10 overs. Marcus Stoinis provided Australia with the breakthrough, getting a nick off Marumani that was caught behind. Cameron Green was the only other bowler amongs wickets, while the rest were put to the sword by Zimbabwe batters.

The defeat is another setback for an Australian team that has been left reeling with injuries and fitness issues before and during the World Cup.

Zimbabwe put up a commanding total on the board | Innings Highlights | T20WC 2026

ICC Men's T20 World Cup, 2026NewsAustralia vs Zimbabwe - Match 19 - 2/13/2026Canada vs United Arab Emirates - Match 20 - 2/13/2026USA vs Netherlands - Match 21 - 2/13/2026 [...] Zimbabwe were rewarded for their disciplined bowling. Zimbabwe pacers Blessing Muzarabani and Brad Evans razed the Aussie top order, then returned to deliver the knockout blow. Muzarabani finished with 4/17, the best bowling figures by a Zimbabwe player at a men's T20 World Cup and the best figures for Zimbabwe against a full member team. His pace partner Evans claimed 3/23 in 3.3 overs.

Their commitment in the field was epitomised by Tony Munyonga diving catch to send back Ben Dwarshuis.

Zimbabwe all over Australia | Powerplay Highlights | T20WC 2026

Zimbabwe bowlers were on fire to grab four Australian wickets in the opening six overs in Colombo.
****************************************************************************************************
len(result['relevant_docs'])
5
 